In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import polars as pl
import polars.selectors as cs

In [ ]:
CWD = Path().resolve()
BASE = CWD

print(f"Folder where groups (data) folder should be: {BASE}")

In [ ]:
groups_map = {
    "plants_plt": ["solanaceae"],
    "plants_mit": ["solanaceae"],
}

In [ ]:
CORE_GAP_MAX = 0.5  # a column is "core" if < 50% of sequences have a gap there
HOMOPOLYMER_MIN = 5  # mononucleotide run length counted as a homopolymer
BASES = list("ACGT")
RUN_NAME = "clustalo_2026-09-30"

In [ ]:
def read_aln(path):
    """Aligned FASTA -> (ids, 2D array of characters)."""
    ids, seqs = [], []
    for line in open(path):
        line = line.strip()
        if line.startswith(">"):
            ids.append(line[1:].split()[0])
            seqs.append([])
        elif line:
            seqs[-1].append(line.upper())
    return ids, np.array([list("".join(s)) for s in seqs])


def pairwise_identity(M):
    """Identity for every pair of rows, ignoring positions where either has a gap."""
    notgap = M != "-"
    out = []
    for i in range(len(M) - 1):
        both = notgap[i] & notgap[i + 1 :]
        same = ((M[i] == M[i + 1 :]) & both).sum(axis=1)
        n = both.sum(axis=1)
        out.append(same[n > 0] / n[n > 0])
    return np.concatenate(out) if out else np.array([])


def in_homopolymer(s, pos):
    """True if position pos of string s sits in a run of >= HOMOPOLYMER_MIN identical bases."""
    if not 0 <= pos < len(s):
        return False

    base = s[pos]
    start = end = pos
    while start > 0 and s[start - 1] == base:
        start -= 1
    while end < len(s) - 1 and s[end + 1] == base:
        end += 1

    return end - start + 1 >= HOMOPOLYMER_MIN


def _gap_runs(s):
    """(start, end) of every run of '-' in s, end exclusive."""
    runs, start = [], None
    for k, ch in enumerate(s + "x"):
        if ch == "-" and start is None:
            start = k
        elif ch != "-" and start is not None:
            runs.append((start, k))
            start = None
    return runs


def aln_metrics(ids, M):
    """Return (summary dict, list of per-sequence dicts) for one alignment."""
    n, L = M.shape
    notgap = M != "-"
    seqs = ["".join(r) for r in M]
    ungapped = [s.replace("-", "") for s in seqs]

    # --- per column: gap fraction, entropy, number of states, parsimony-informative
    gapf = 1 - notgap.mean(axis=0)
    ent = np.full(L, np.nan)
    nstates = np.zeros(L, int)
    pis = np.zeros(L, bool)
    for j in range(L):
        c = Counter(b for b in M[:, j] if b in BASES)
        if not c:
            continue
        p = np.array(list(c.values())) / sum(c.values())
        ent[j] = -(p * np.log2(p)).sum()
        nstates[j] = len(c)
        pis[j] = sum(v >= 2 for v in c.values()) >= 2
    core = gapf < CORE_GAP_MAX
    var = core & (nstates > 1)

    # --- pairwise identity (all sequences, and unique sequences only)
    pid = pairwise_identity(M)
    uniq = list({s: i for i, s in enumerate(ungapped)}.values())
    pid_u = pairwise_identity(M[uniq])

    # --- longest run of columns that are gap-free and identical in every sequence
    longest = run = 0
    for ok in (gapf == 0) & (nstates == 1):
        run = run + 1 if ok else 0
        longest = max(longest, run)

    # --- indels: internal gap runs (terminal gaps are boundary differences, not indels)
    first = notgap.argmax(axis=1)
    last = L - 1 - notgap[:, ::-1].argmax(axis=1)
    events, homo_events, gaps_per_seq = set(), set(), []
    for i, s in enumerate(seqs):
        runs = [(a, b) for a, b in _gap_runs(s) if a > first[i] and b <= last[i]]
        gaps_per_seq.append(len(runs))
        for a, b in runs:
            events.add((a, b))
            left = s[:a].replace("-", "")
            ctx = left + s[b:].replace("-", "")
            if in_homopolymer(ctx, len(left) - 1) or in_homopolymer(ctx, len(left)):
                homo_events.add((a, b))

    # --- boundaries: does each sequence start/end at the most common column?
    off = (first != Counter(first).most_common(1)[0][0]) | (
        last != Counter(last).most_common(1)[0][0]
    )

    summary = {
        "aln_n_seqs": n,
        "aln_len": L,
        "core_len": int(core.sum()),
        "n_haplotypes": len(set(ungapped)),
        "mean_pid": pid.mean() if len(pid) else None,
        "min_pid": pid.min() if len(pid) else None,
        "mean_pid_uniq": pid_u.mean() if len(pid_u) else None,
        "var_sites": int(var.sum()),
        "pis": int((core & pis).sum()),
        "invariant_frac": 1 - var.sum() / core.sum() if core.any() else None,
        "mean_entropy": float(np.nanmean(ent[core])) if core.any() else None,
        "indel_events": len(events),
        "indels_per_seq": float(np.mean(gaps_per_seq)),
        "longest_invariant_block": longest,
        "homopolymer_indel_frac": len(homo_events) / len(events) if events else None,
        "boundary_offset_frac": float(off.mean()),
        "gc_mean": float(
            np.mean([(u.count("G") + u.count("C")) / len(u) for u in ungapped if u])
        ),
    }

    # --- per sequence: identity to the majority consensus
    cons = np.array(
        [
            (
                Counter(b for b in M[:, j] if b in BASES).most_common(1)[0][0]
                if nstates[j]
                else "-"
            )
            for j in range(L)
        ]
    )
    per_seq = []
    for i, row in enumerate(M):
        mask = notgap[i] & (cons != "-")
        per_seq.append(
            {
                "seq_id": ids[i],
                "ungapped_len": len(ungapped[i]),
                "id_to_consensus": (
                    float((row[mask] == cons[mask]).mean()) if mask.any() else None
                ),
                "internal_gaps": gaps_per_seq[i],
                "boundary_offset": bool(off[i]),
            }
        )
    return summary, per_seq

In [ ]:
for g, subgs in groups_map.items():
    for subg in subgs:
        subg_dir = BASE / g / subg
        aln_root = subg_dir / "alignments"
        run_dir = aln_root / RUN_NAME
        complete_out = aln_root / f"{RUN_NAME}_gp_complete.tsv"
        per_seq_out = aln_root / f"{RUN_NAME}_aln.tsv"
        print(f"Processing {g}/{subg} [{RUN_NAME}]")

        gp_file = subg_dir / "gene_pairs_filtered.tsv"
        if not gp_file.exists():
            print(f"  no {gp_file.name}, skipping")
            continue
        if not run_dir.exists():
            print(f"  no run folder {run_dir.relative_to(BASE)}, skipping")
            continue

        gp_df = pl.read_csv(gp_file, separator="\t")

        summaries, per_seq_rows, missing = [], [], []
        for gp in gp_df["gene_pair"]:
            aln_file = run_dir / f"gp_{gp}_aligned.fasta"
            if not aln_file.exists():
                missing.append(gp)
                continue

            ids, M = read_aln(aln_file)
            summary, per_seq = aln_metrics(ids, M)
            summaries.append({"gene_pair": gp, **summary})
            per_seq_rows += [{"gene_pair": gp, **r} for r in per_seq]
            print(f"  {gp}: n={summary['aln_n_seqs']} mean_pid={summary['mean_pid']}")

        if missing:
            print(f"  missing alignments ({len(missing)}): {missing}")
        if not summaries:
            print("  no alignments found, skipping outputs")
            continue

        metrics = pl.DataFrame(summaries, infer_schema_length=None)
        complete = gp_df.join(metrics, on="gene_pair", how="left")
        complete = complete.with_columns(cs.float().round(4))
        complete.write_csv(complete_out, separator="\t")

        per_seq_df = pl.DataFrame(per_seq_rows, infer_schema_length=None)
        per_seq_df = per_seq_df.with_columns(cs.float().round(4))
        per_seq_df.write_csv(per_seq_out, separator="\t")

        print(f"  Wrote {complete_out.relative_to(BASE)} ({complete.height} rows)")
        print(f"  Wrote {per_seq_out.relative_to(BASE)} ({per_seq_df.height} rows)")